In [ ]:

from config.config import *
from pyspark.sql import functions as F

tables = [
    "dim_country", "dim_date", "fact_tourism_arrivals", "fact_nights_spent",
    "fact_hotel_performance", "fact_hotel_occupancy",
    "gold_yearly_arrivals_by_country", "gold_monthly_arrivals_by_country",
    "gold_tourism_growth_yearly", "gold_tourism_growth_monthly",
    "gold_destination_rankings", "gold_tourism_seasonality",
    "gold_nights_spent_trends", "gold_hotel_capacity_trends",
    "gold_hotel_occupancy_trends", "gold_visitor_category_analysis",
    "gold_country_comparison", "gold_source_comparison",
]

for t in tables:
    try:
        df = spark.read.format("delta").load(f"{gold_path}/{t}")
        print(f"TABLE {t}: rows={df.count()}")
        dup = df.groupBy(*df.columns).count().filter(F.col("count") > 1).count()
        print(f"  duplicate grain groups: {dup}")
    except Exception as e:
        print(f"TABLE {t}: ERROR {e}")

# Validate fact_tourism_arrivals: no negative standardized_value, no zero where allowed
fa = spark.read.format("delta").load(f"{gold_path}/fact_tourism_arrivals")
print("fact_tourism_arrivals negatives:", fa.filter(F.col("standardized_value") < 0).count())
print("fact_tourism_arrivals nulls in standardized_value:", fa.filter(F.col("standardized_value").isNull()).count())

# Validate fact_hotel_occupancy: no negatives
occ = spark.read.format("delta").load(f"{gold_path}/fact_hotel_occupancy")
print("fact_hotel_occupancy negatives:", occ.filter(F.col("occupancy_rate") < 0).count())


In [ ]:
from pyspark.sql import functions as F

# Full-grain duplicate validation (new cell added for Power BI Phase)
fact_arr = spark.read.format("delta").load(f"{gold_path}/fact_tourism_arrivals")
fact_nights = spark.read.format("delta").load(f"{gold_path}/fact_nights_spent")
dim_country = spark.read.format("delta").load(f"{gold_path}/dim_country")

# Duplicates with full fact grains
arr_dups = (
    fact_arr.groupBy("country_key", "geo_code", "year", "month", "date_key", "frequency",
                     "source_system", "indicator_code", "visitor_category",
                     "residence_category", "accommodation_category", "observation_status")
    .count().filter(F.col("count") > 1)
    .count()
)
nights_dups = (
    fact_nights.groupBy("country_key", "geo_code", "year", "month", "date_key",
                        "residence_category", "activity_category", "source_system")
    .count().filter(F.col("count") > 1)
    .count()
)
print("fact_tourism_arrivals full-grain duplicates:", arr_dups)
print("fact_nights_spent full-grain duplicates:", nights_dups)

# Nulls by source
fact_arr.groupBy("source_system").agg(
    F.count("*").alias("rows"),
    F.count(F.when(F.col("standardized_value").isNull(), 1)).alias("nulls")
).orderBy("source_system").show(truncate=False)
fact_nights.groupBy("source_system").agg(
    F.count("*").alias("rows"),
    F.count(F.when(F.col("standardized_value").isNull(), 1)).alias("nulls")
).orderBy("source_system").show(truncate=False)

# Growth nulls
for t in ["gold_tourism_growth_yearly", "gold_tourism_growth_monthly"]:
    df = spark.read.format("delta").load(f"{gold_path}/{t}")
    print(f"{t}: null growth_pct =", df.filter(F.col("growth_pct").isNull()).count())

# dim_country canonicalization validation
print("dim_country rows:", dim_country.count())
print("dim_country duplicate country_keys:",
      dim_country.groupBy("country_key").count().filter(F.col("count") > 1).count())
